# 공통23종 Multi-FraudGT 재학습

단순0 대체와 달리 학습 입력부터11종을 제외한 fresh 학습입니다. 이 노트북은 보존된 학습 코드와284에폭 기록을 검토합니다. 전체 학습에는 원본 환경 복원이 필요합니다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "saml_d_experiments"]
ROOT = next((p.resolve() for p in candidates if (p / "PACKAGE.json").exists()
             and json.loads((p / "PACKAGE.json").read_text())["package"] == "saml_d_experiments"), None)
if ROOT is None:
    raise FileNotFoundError("압축을 풀고 해당 패키지의 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))
print("패키지:", ROOT)


In [ ]:
RESULT = ROOT/'results/common23'
plan = json.loads((RESULT/'plan.json').read_text())
display(pd.DataFrame([plan['training']]))
print('completed budget:',plan['completed_epoch_budget'],'LR horizon:',plan['lr_schedule_horizon'])
print('initialization:', plan['initialization'])
display(pd.read_csv(RESULT/'피처_포함제외.csv'))

## 실제 구현

아래 셀은 보존된 Python 코드를 보여줍니다. 모델 코드를 import하거나 현재 학습 프로세스를 조작하지 않습니다. GPU 입력 조회는 무손실 최적화이며 CPU 그래프 샘플러는 유지했습니다.

In [ ]:
from IPython.display import Code
SOURCE = ROOT/'reference_source/Multi_FraudGT_SAML_공통23피처_재학습_20261109'
display(Code((SOURCE/'common_model.py').read_text(), language='python'))
display(Code((SOURCE/'train.py').read_text(), language='python'))

## 학습률·학습/검증 곡선

중간 Validation은 official readout 표본입니다. 아래 최고 F1을 최종 전체 Test F1과 혼동하지 않습니다.

In [ ]:
history = pd.read_csv(RESULT/'training/epoch_metrics.csv')
selection = json.loads((RESULT/'frozen_selection.json').read_text())
print('selected epoch:', selection['selected_epoch'])
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,3,figsize=(14,3.5))
axes[0].plot(history.epoch,history.learning_rate); axes[0].set(ylabel='Learning rate')
axes[1].plot(history.epoch,history.train_loss); axes[1].set(ylabel='Train loss')
v=history.dropna(subset=['val_F1'])
axes[2].plot(v.epoch,100*v.val_F1);axes[2].axvline(selection['selected_epoch'],color='red',alpha=.5);axes[2].set(ylabel='Sampled Val F1 (%)')
for ax in axes:ax.set_xlabel('Epoch');ax.grid(alpha=.2)
fig.tight_layout();plt.show()
display(pd.DataFrame([json.loads((RESULT/'health_summary.json').read_text())]))

In [ ]:
display(pd.read_csv(RESULT/'전체_비교지표.csv'))
display(pd.read_csv(RESULT/'evaluation/saml/pattern_metrics.csv'))

## 전체 재실행 진입점

외부 원본 환경을 별도 실험 디렉터리에 복원하고 plan/manifest를 새로 고정한 뒤 `preflight.py → train.py → evaluate.py hi_test → evaluate.py saml → report.py`를 실행합니다. `run_pipeline.py`가 같은 순서를 관리합니다. 세부 외부 파일은 [재실행 문서](../docs/04_코드_재실행.md)를 확인합니다. 이 ZIP에는 원본 거래·가중치·그래프 캐시가 없으므로 기본 셀 실행을 재학습이라고 부르지 않습니다.